# SHAP 산출물 빌드 (Colab 단독 실행)

ZITboost 5-fold 모델에서 μ/π 컴포넌트별 SHAP을 계산하고 unit-level mean으로 집계하여 parquet/json/csv로 저장.

## 입력 (Drive)
- `code.zip` (ID `1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I`) — `utils/`, `setup.py`
- `dataset.zip` (ID `1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO`) — CSV 4개
- **별도 업로드 필요**: `fold_models.pkl`, `oof_die.csv`, `3_modeling/final/modules/` 일체

## 출력 → `/content/project/5_dashboard/data/model/`
- `shap_mu_unit.parquet`
- `shap_pi_unit.parquet`
- `shap_base.json`
- `shap_summary.csv`

마지막 셀에서 4개 파일을 zip으로 묶어 다운로드. 로컬 `5_dashboard/data/model/`에 풀어 넣으면 끝.


## 1. 환경 준비 — code.zip + dataset.zip 받기

In [ ]:
import os, sys

os.system('pip install -q gdown shap')
os.makedirs('/content/project', exist_ok=True)

# 1) 코드 (utils/, setup.py)
if not os.path.exists('/content/project/utils'):
    os.system('gdown 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip')
    os.system('unzip -qo /content/code.zip -d /content/project')

# 2) 데이터 (CSV 4개)
if not os.path.exists('/content/project/0_data/compet_xs_data.csv'):
    os.makedirs('/content/project/0_data', exist_ok=True)
    os.system('gdown 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip')
    os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
    os.remove('/content/project/0_data/dataset.zip')

sys.path.insert(0, '/content/project')
sys.path.insert(0, '/content/project/3_modeling')
%cd /content/project
!ls

## 2. 추가 업로드 — fold_models.pkl, oof_die.csv, 3_modeling/final/modules/

`code.zip`에 안 들어 있는 파일들. **세 가지 방법** 중 편한 것 선택.

### 방법 A: 직접 업로드 (가장 간단, 작은 파일에 OK)
아래 셀 실행 후 4개 항목을 한꺼번에 선택해서 업로드:
- `fold_models.pkl` (15MB)
- `oof_die.csv` (11MB)
- `3_modeling/final/modules/` 폴더를 zip으로 압축한 파일 (`modules.zip`)

### 방법 B: Google Drive 마운트 (재실행 잦으면 추천)
프로젝트 폴더를 통째로 Drive에 올린 뒤 마운트.

### 방법 C: 각자 gdown
각 파일을 Drive에 올려 ID 받아서 gdown.

**아래 셀은 방법 A 기준**. B/C 쓰면 이 셀은 스킵하고 직접 경로 맞춰 배치.

In [ ]:
# 방법 A: files.upload() — fold_models.pkl, oof_die.csv, modules.zip 한 번에 선택
from google.colab import files
import shutil, zipfile

os.makedirs('/content/project/4_output/final/zit_only', exist_ok=True)
os.makedirs('/content/project/3_modeling/final', exist_ok=True)

uploaded = files.upload()  # 브라우저 파일 다이얼로그

for fname in uploaded:
    src = f'/content/{fname}'
    if fname == 'fold_models.pkl':
        shutil.move(src, '/content/project/4_output/final/zit_only/fold_models.pkl')
    elif fname == 'oof_die.csv':
        shutil.move(src, '/content/project/4_output/final/zit_only/oof_die.csv')
    elif fname == 'modules.zip':
        # zip 안 구조를 자동 감지하여 `from final.modules import ...` 가 동작하게 배치
        with zipfile.ZipFile(src) as z:
            names = z.namelist()
            if any(n.startswith('final/modules/') for n in names):
                z.extractall('/content/project/3_modeling')
            elif any(n.startswith('modules/') for n in names):
                z.extractall('/content/project/3_modeling/final')
            else:
                z.extractall('/content/project/3_modeling/final/modules')
        os.remove(src)
    else:
        print(f'알 수 없는 파일 (스킵): {fname}')

# __init__.py 보장 (import path 확보)
for d in ['/content/project/3_modeling',
          '/content/project/3_modeling/final',
          '/content/project/3_modeling/final/modules']:
    init = os.path.join(d, '__init__.py')
    if not os.path.exists(init):
        open(init, 'w').close()

print('
배치 결과:')
for p in ['/content/project/4_output/final/zit_only/fold_models.pkl',
          '/content/project/4_output/final/zit_only/oof_die.csv',
          '/content/project/3_modeling/final/modules/preprocess.py',
          '/content/project/3_modeling/final/modules/hpo.py']:
    print(f'  {"✓" if os.path.exists(p) else "✗"} {p}')

## 3. SHAP 계산 + 저장

`build_model_artifacts.py`의 `build_shap_artifacts()` 로직을 그대로 옮긴 단일 셀.

In [ ]:
import json, time
from pathlib import Path
import numpy as np
import pandas as pd
import pickle
import shap

from final.modules import preprocess as _pp
from final.modules.hpo import _make_unit_folds
from utils.data import load_all, get_feat_cols, split_xs
from utils.config import KEY_COL, TARGET_COL

# ─── 설정 ─────────────────────────────────────────────────
SEED = 42
N_FOLDS = 5
SHAP_PARAMS = {}              # 학습 시점과 동일 (DEFAULT_PARAMS)
SHAP_CLIP_Y_EXTREME = True    # 01_zit_only.ipynb 동일
SHAP_RMSE_TOL = 1e-6          # OOF die 재예측 허용 오차

PROJECT_ROOT = Path('/content/project')
FM_PATH = PROJECT_ROOT / '4_output/final/zit_only/fold_models.pkl'
OOF_DIE_PATH = PROJECT_ROOT / '4_output/final/zit_only/oof_die.csv'
OUT_DIR = PROJECT_ROOT / '5_dashboard/data/model'
OUT_DIR.mkdir(parents=True, exist_ok=True)

def log(m): print(f'[shap] {m}', flush=True)
t0 = time.time()

# ─── pkl 로드 ─────────────────────────────────────────────
with FM_PATH.open('rb') as f:
    fm = pickle.load(f)
feature_names = list(fm['feature_names'])
fold_models = fm['fold_models']
n_feat = len(feature_names)
log(f'pkl 로드: model_name={fm["model_name"]}, n_feat={n_feat}, n_folds={len(fold_models)}')

# ─── [5-1] preprocess.run 재실행 ───────────────────────────
log('[5-1] load_all + preprocess.run (PARAMS={})')
xs, ys = load_all()
feat_cols_raw = get_feat_cols(xs)
xs_dict = split_xs(xs)

ys_input = {k: v.copy() for k, v in ys.items()}
if SHAP_CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = int((y_raw >= 1.0).sum())
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    log(f'  CLIP_Y_EXTREME: {y_raw.max():.6f} → {second_max:.6f} ({n_clipped}개)')

pp = _pp.run(xs, ys_input, feat_cols_raw, xs_dict, params=SHAP_PARAMS)
xs_train, xs_val, xs_test = pp['xs_train'], pp['xs_val'], pp['xs_test']
feat_cols_clean = pp['feat_cols']
log(f'  전처리 완료: train={xs_train.shape}, val={xs_val.shape}, test={xs_test.shape}, feat={len(feat_cols_clean)}')

# ─── [5-2] 검증 가드 ──────────────────────────────────────
log('[5-2] 검증 가드: feat_cols 일치 + OOF die 재현')
if feat_cols_clean != feature_names:
    diff_a = set(feat_cols_clean) - set(feature_names)
    diff_b = set(feature_names) - set(feat_cols_clean)
    raise RuntimeError(
        f'feat_cols 불일치 — 추가({len(diff_a)}): {sorted(diff_a)[:5]}, '
        f'누락({len(diff_b)}): {sorted(diff_b)[:5]}'
    )

X_train = xs_train[feature_names].values
X_val   = xs_val[feature_names].values
X_test  = xs_test[feature_names].values
log(f'  X shape: train={X_train.shape}, val={X_val.shape}, test={X_test.shape}')

train_units = ys_input['train'][KEY_COL].unique()
folds = _make_unit_folds(train_units, N_FOLDS, SEED)

n_tr_die = len(xs_train)
oof_pred_die = np.full(n_tr_die, np.nan)
for i, ((_tr, vl_units), model) in enumerate(zip(folds, fold_models)):
    vl_mask = xs_train[KEY_COL].isin(set(vl_units)).values
    oof_pred_die[vl_mask] = model.predict(X_train[vl_mask])
    log(f'    fold {i+1}/{N_FOLDS} predict done')
if np.isnan(oof_pred_die).any():
    raise RuntimeError('OOF 재예측에 NaN — fold split 재현 실패')

if OOF_DIE_PATH.exists():
    oof_die_saved = pd.read_csv(OOF_DIE_PATH, usecols=['pred'])
    if len(oof_die_saved) != n_tr_die:
        raise RuntimeError(f'oof_die.csv 행수 불일치: {len(oof_die_saved)} vs {n_tr_die}')
    diff_max = float(np.max(np.abs(oof_die_saved['pred'].values - oof_pred_die)))
    log(f'  oof_die pred max abs diff = {diff_max:.2e}')
    if diff_max > SHAP_RMSE_TOL:
        raise RuntimeError(
            f'OOF 재예측 불일치 (diff={diff_max:.2e} > tol={SHAP_RMSE_TOL}). '
            'SHAP_PARAMS({})가 학습 시점과 다를 가능성 — study_meta 확인 필요.'
        )
    log('  ✓ OOF die 예측 일치')
else:
    log(f'  ⚠ {OOF_DIE_PATH.name} 없음 — 가드 스킵')

# ─── [5-3] SHAP 계산 (5-fold 평균) ────────────────────────
log('[5-3] TreeExplainer (lgb_mu, lgb_pi) × 5 fold')
shap_mu_train = np.zeros((n_tr_die, n_feat), dtype=np.float32)
shap_mu_val   = np.zeros((len(xs_val),  n_feat), dtype=np.float32)
shap_mu_test  = np.zeros((len(xs_test), n_feat), dtype=np.float32)
shap_pi_train = np.zeros_like(shap_mu_train)
shap_pi_val   = np.zeros_like(shap_mu_val)
shap_pi_test  = np.zeros_like(shap_mu_test)
mu_base, pi_base = 0.0, 0.0

for i, model in enumerate(fold_models):
    ex_mu = shap.TreeExplainer(model.lgb_mu_)
    ex_pi = shap.TreeExplainer(model.lgb_pi_)
    shap_mu_train += ex_mu.shap_values(X_train).astype(np.float32) / N_FOLDS
    shap_mu_val   += ex_mu.shap_values(X_val).astype(np.float32) / N_FOLDS
    shap_mu_test  += ex_mu.shap_values(X_test).astype(np.float32) / N_FOLDS
    shap_pi_train += ex_pi.shap_values(X_train).astype(np.float32) / N_FOLDS
    shap_pi_val   += ex_pi.shap_values(X_val).astype(np.float32) / N_FOLDS
    shap_pi_test  += ex_pi.shap_values(X_test).astype(np.float32) / N_FOLDS
    mu_base += float(np.asarray(ex_mu.expected_value).ravel()[0]) / N_FOLDS
    pi_base += float(np.asarray(ex_pi.expected_value).ravel()[0]) / N_FOLDS
    log(f'    fold {i+1}/{N_FOLDS} SHAP done')

# ─── [5-4] die→unit mean 집계 ─────────────────────────────
log('[5-4] die→unit mean 집계')
def _die_to_unit(xs_split, shap_die):
    df = pd.DataFrame(shap_die, columns=feature_names)
    df[KEY_COL] = xs_split[KEY_COL].values
    return df.groupby(KEY_COL, sort=False)[feature_names].mean().reset_index()

mu_units, pi_units = [], []
for split_name, xs_split, sh_mu, sh_pi in [
    ('train',      xs_train, shap_mu_train, shap_pi_train),
    ('validation', xs_val,   shap_mu_val,   shap_pi_val),
    ('test',       xs_test,  shap_mu_test,  shap_pi_test),
]:
    mu_u = _die_to_unit(xs_split, sh_mu); mu_u['split'] = split_name
    pi_u = _die_to_unit(xs_split, sh_pi); pi_u['split'] = split_name
    mu_units.append(mu_u); pi_units.append(pi_u)
shap_mu_unit = pd.concat(mu_units, ignore_index=True)
shap_pi_unit = pd.concat(pi_units, ignore_index=True)
log(f'  unit shape: mu={shap_mu_unit.shape}, pi={shap_pi_unit.shape}')

# ─── [5-5] 저장 ───────────────────────────────────────────
log('[5-5] 저장')
shap_mu_unit.to_parquet(OUT_DIR / 'shap_mu_unit.parquet', index=False)
shap_pi_unit.to_parquet(OUT_DIR / 'shap_pi_unit.parquet', index=False)

with (OUT_DIR / 'shap_base.json').open('w', encoding='utf-8') as f:
    json.dump({'mu_base': mu_base, 'pi_base': pi_base}, f, indent=2)

summary = pd.DataFrame({
    'feature': feature_names,
    'mean_abs_shap_mu': np.abs(shap_mu_unit[feature_names].values).mean(axis=0),
    'mean_abs_shap_pi': np.abs(shap_pi_unit[feature_names].values).mean(axis=0),
})
summary['mean_abs_shap_total'] = summary['mean_abs_shap_mu'] + summary['mean_abs_shap_pi']
summary = summary.sort_values('mean_abs_shap_total', ascending=False).reset_index(drop=True)
summary.to_csv(OUT_DIR / 'shap_summary.csv', index=False)
log(f'  Top 5: {", ".join(summary["feature"].head(5).tolist())}')
log(f'전체 완료 ({time.time()-t0:.1f}s) → {OUT_DIR}')
!ls -la {OUT_DIR}

## 4. 산출물 다운로드

4개 파일을 zip으로 묶어서 다운로드. 로컬 `5_dashboard/data/model/`에 풀어 넣으면 끝.

In [ ]:
import shutil
from google.colab import files

zip_path = '/content/shap_artifacts.zip'
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', OUT_DIR)
print(f'zip 생성: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)')
files.download(zip_path)